# Part C — Prompt Engineering Experiments
**CS4241 | Student: Farima Konaré | Index: 10012200004**

This notebook compares the three prompt templates on the same query and analyses the differences in output.

Templates:
1. Baseline (minimal instructions)
2. Hallucination-controlled (strict grounding)
3. With memory (grounding + conversation history)

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

from retrieval.embedder import get_embedder
from retrieval.vector_store import VectorStore
from retrieval.retriever import HybridRetriever
from generation.prompt_builder import build_prompt
from generation.llm_client import generate
from ingestion.chunker import load_chunks
from memory import ConversationMemory

chunks = load_chunks('all_chunks.json')
emb = get_embedder()
store = VectorStore(dim=emb.dim)
store.load()
retriever = HybridRetriever(store, chunks)
print('Setup complete')

Index loaded: 1565 vectors from /Users/mac/Desktop/ai_10012200004/experiments/../src/retrieval/../../data/processed/faiss.index
Setup complete


In [2]:
TEST_QUERY = 'Who won the 2020 presidential election in Ghana?'

q_vec = emb.encode_query(TEST_QUERY)
results = retriever.retrieve(TEST_QUERY, q_vec, k=5)

In [3]:
# Template 1 — Baseline
prompt1 = build_prompt(TEST_QUERY, results, template_id=1)
response1 = generate(prompt1)
print('=== Template 1 (Baseline) ===')
print(response1)
print()

=== Template 1 (Baseline) ===
The provided context does not contain information about the overall winner of the 2020 Ghana presidential election. It only provides vote counts for specific candidates in certain regions.



In [4]:
# Template 2 — Hallucination-controlled
prompt2 = build_prompt(TEST_QUERY, results, template_id=2)
response2 = generate(prompt2)
print('=== Template 2 (Hallucination-Controlled) ===')
print(response2)
print()

=== Template 2 (Hallucination-Controlled) ===
I don't have enough information in the knowledge base to answer this question.



In [5]:
# Template 3 — With memory (simulate prior turn)
mem = ConversationMemory()
mem.add_turn('What were the key issues in the 2020 Ghana election?',
             'The 2020 election focused on COVID-19 management, economy, and infrastructure.')

prompt3 = build_prompt(TEST_QUERY, results, memory_text=mem.format_for_prompt(), template_id=3)
response3 = generate(prompt3)
print('=== Template 3 (With Memory) ===')
print(response3)
print()

=== Template 3 (With Memory) ===
The retrieved context is insufficient to determine the winner of the 2020 presidential election in Ghana. The provided information only includes vote counts for specific candidates in certain regions, but does not provide a comprehensive overview of the election results or the overall winner.



## Analysis

| Template | Response style | Hallucination risk | Memory use | Verdict |
|---|---|---|---|---|
| 1 — Baseline | Cautious: says context is insufficient | Low (in this run) | No | Acceptable single-turn fallback |
| 2 — Controlled | Explicit refusal when evidence is insufficient | Lowest | No | Strongest hallucination control |
| 3 — With memory | Also refuses unsupported claim, with more explanation | Low | Yes | Best chat UX with grounding |

**Key observation from this run:** all three templates avoided inventing a winner. Template 2 was the most concise refusal; Template 3 was the most explanatory while staying grounded.

Template 3 remains the production default because this project is a conversational app and memory-aware responses are more coherent across turns.

In [6]:
# Context window management demo
from generation.prompt_builder import manage_context_window, MAX_CONTEXT_CHARS

before = len(results)
selected = manage_context_window(results)
after = len(selected)
total_chars = sum(len(r['chunk']['text']) for r in selected)

print(f'Chunks before window management: {before}')
print(f'Chunks after (budget={MAX_CONTEXT_CHARS} chars): {after}')
print(f'Total context chars used: {total_chars}')

Chunks before window management: 5
Chunks after (budget=8000 chars): 5
Total context chars used: 605
